# Experimentos de Pi-zero (pi0) sobre LIBERO

Corre pi0 (LeRobot) en varias tareas de LIBERO y muestra los resultados (tabla,
tasa de éxito y videos). Es el mismo framework que el notebook de OpenVLA
(`core`, `benchmarks`, `models`); aquí solo cambia el **modelo**.

> Requiere el entorno `pizero` (ver `requirements/environment-pizero.yml`), con **GPU** y el
> checkpoint `lerobot/pi0_libero_finetuned` descargado en `HF_HOME`.


## 1. Setup


In [ ]:
import os, sys, glob

# Opcion A (frontera de proceso): este notebook es el CLIENTE del benchmark y
# corre en el entorno `openvla` (robosuite 1.4.1, numpy<2). pi0 (lerobot/torch)
# vive en OTRO proceso (models/Pi_zero/server.py, entorno `pizero`) y se consume
# por HTTP. Por eso aqui NO se instala ni se importa torch/lerobot, y no hace
# falta el shim de torch.load: eso lo hace el servidor al cargar el checkpoint.

# El notebook (benchmark) corre SIN GPU: la unica tarjeta la usa en EXCLUSIVA el
# servidor pi0. Por eso el render headless de MuJoCo/LIBERO va por OSMesa
# (software/CPU), NO por EGL, que exige una GPU (y ademas revienta si
# CUDA_VISIBLE_DEVICES=""). Debe fijarse ANTES de importar robosuite/LIBERO, que
# lee MUJOCO_GL al importarse. Necesita libOSMesa en el env (conda-forge mesalib).
os.environ.setdefault("MUJOCO_GL", "osmesa")   # render headless por CPU (sin GPU)


def _find_project_root():
    """Sube desde el cwd buscando la raiz del proyecto (simulation.py + core/)."""
    d = os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(d, "simulation.py")) and \
           os.path.isdir(os.path.join(d, "core")):
            return d
        d = os.path.dirname(d)
    hits = glob.glob(os.path.join(os.getcwd(), "**", "simulation.py"), recursive=True)
    if hits:
        return os.path.dirname(os.path.abspath(hits[0]))
    raise RuntimeError("No encontre la raiz del proyecto (simulation.py + core/).")


ROOT = _find_project_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
os.environ.setdefault("HF_HOME", os.path.join(ROOT, "hf_cache"))   # cache de pesos
print("Proyecto:", ROOT)
print("HF_HOME :", os.environ["HF_HOME"])
print("MUJOCO_GL:", os.environ["MUJOCO_GL"], "| CUDA_VISIBLE_DEVICES:", repr(os.environ.get("CUDA_VISIBLE_DEVICES")))

## 2. Configuración


In [2]:
from dataclasses import dataclass


@dataclass
class Config:
    suite: str = "libero_10"
    num_tasks: int = 3             # cuantos escenarios (tareas) probar
    episodes_per_task: int = 1     # configuraciones iniciales por tarea
    max_steps: int = 520           # libero_10 es de horizonte largo (oficial ~520)
    out_dir: str = "output/experiments_pizero"


cfg = Config()
cfg


Config(suite='libero_10', num_tasks=3, episodes_per_task=1, max_steps=520, out_dir='output/experiments_pizero')

## 3. Cargar el modelo (una sola vez)


In [3]:
from models.serving import RemoteModel

# pi0 corre como servidor aparte (lo levanta scriptExperiment.sh con MODEL=pi0).
# El notebook es solo cliente: misma interfaz Model, sin deps del modelo.
model = RemoteModel(url="http://localhost:9000", timeout=600)   # 9000 = servidor de pi0
print("cliente conectado ->", model.url)


cliente conectado -> http://localhost:9000


## Experimento: barrido por pasos del entorno

En **cada estado del entorno** se generan `N_SAMPLES` muestras del modelo (con el
mismo input), cada una con su trayectoria de flow-matching en el CSV. Luego el
entorno se **avanza ejecutando el chunk completo** de acciones que genera pi0
(todas las acciones del muestreo, una a una) y se repite.

El CSV `output/pi0_denoise/denoise_log.csv` lleva la columna **`step`** (el paso
del entorno), que el cliente fija con `model.set_context(step=...)`. Cada corrida
sigue teniendo su `call` (global) e `iteration` (0 = ruido inicial).

> El CSV se **reinicia al arrancar el servidor**; si corriste otro experimento en
> la misma sesion del servidor, este barrido se **anexa**. Para un CSV limpio,
> relanza el job antes de correr esto.

In [ ]:
# === Barrido: por cada paso del entorno, N_SAMPLES muestras del modelo =========
# En cada estado: N_SAMPLES corridas con la MISMA observacion (se loguea la
# trayectoria de denoising con columna `step`). Luego se avanza el entorno
# ejecutando el CHUNK COMPLETO de acciones que genera pi0 (no solo la primera)
# y se repite N_STEPS veces.
import numpy as np

# Un entorno (reusa uno si ya existe).
try:
    probe = benchmarks[0]
except NameError:
    from benchmarks.libero import LiberoController
    probe = LiberoController(task_id=0, suite=cfg.suite)

N_STEPS = 5         # pasos del barrido (cada uno ejecuta un chunk completo)
N_SAMPLES = 100     # muestras del modelo por paso

obs = probe.reset(episode=0)
print("instruccion:", obs.instruction)

for step in range(N_STEPS):
    model.set_context(step=step)      # el servidor etiqueta el CSV con este step
    last = None
    for i in range(N_SAMPLES):
        model.reset()                 # fuerza re-computo (nuevo ruido de flow-matching)
        last = model.act(obs)         # misma obs -> N_SAMPLES muestras logueadas
    # Avanza el entorno ejecutando el CHUNK COMPLETO generado por pi0 (todas las
    # acciones del ultimo muestreo), una a una, como haria `run_episode`.
    chunk = list(last)
    result = None
    for action in chunk:
        result = probe.step(action)
        obs = result.observation
        if result.done:
            break
    done = bool(result.done) if result is not None else False
    print(f"step {step}: {N_SAMPLES} muestras logueadas | env avanzado {len(chunk)} accion(es) del chunk (done={done})")
    if done:
        print("El entorno termino (done=True); detengo el barrido.")
        break

print("Listo. CSV -> output/pi0_denoise/denoise_log.csv (columna 'step').")

## Experimento con ruido AMPLIO (mismo barrido, otro CSV)

Igual que el barrido anterior, pero el ruido inicial del flow-matching se **escala
por dimensión** según la magnitud típica de los datos del CSV ya generado (en vez
de N(0,1)), de modo que el origen cubra un área mucho mayor.

La escala y la ruta del CSV de salida se envían al servidor **en caliente** con
`model.set_context(...)` — sin archivos nuevos ni relanzar el job. Escribe en
`output/pi0_denoise/denoise_big_noise.csv`; analízalo en `flow_matching_analisis.ipynb`
apuntando `CSV_PATH` a ese archivo.

In [ ]:
# === Barrido con ruido AMPLIO ==================================================
# 1) Escala por dimension = magnitud tipica (|v| medio) de las acciones finales
#    del CSV ya generado. 2) Se envia al servidor con set_context (en caliente).
#    3) Mismo barrido, pero escribiendo a un CSV nuevo.
import numpy as np
import pandas as pd

BASE_CSV = "output/pi0_denoise/denoise_log.csv"        # CSV con ruido N(0,1) normal
BIG_CSV  = "output/pi0_denoise/denoise_big_noise.csv"  # salida con ruido amplio

# 1) Escala por dimension desde las acciones finales del CSV base.
base = pd.read_csv(BASE_CSV)
vcols = [c for c in base.columns if c.startswith("v") and c[1:].isdigit()]
vcols.sort(key=lambda c: int(c[1:]))
fi = base.groupby("call")["iteration"].transform("max")
finals = base[base["iteration"] == fi]
scale = np.abs(finals[vcols].to_numpy(dtype=float)).mean(axis=0)  # (chunk*action_dim,)
print(f"escala: dim={scale.size}  media={scale.mean():.4f}  max={scale.max():.4f}")

# 2) Enviar escala + ruta de salida al servidor (en caliente, via HTTP).
model.set_context(noise_scale=scale.tolist(), csv_path=BIG_CSV)

# 3) Mismo barrido que antes, ahora con ruido amplio -> BIG_CSV. Cada paso avanza
#    el entorno ejecutando el CHUNK COMPLETO de pi0 (no solo la primera accion).
N_STEPS, N_SAMPLES = 5, 100
obs = probe.reset(episode=0)
print("instruccion:", obs.instruction)
for step in range(N_STEPS):
    model.set_context(step=step)
    last = None
    for i in range(N_SAMPLES):
        model.reset()
        last = model.act(obs)
    # Ejecuta el chunk completo (todas las acciones del ultimo muestreo).
    chunk = list(last)
    result = None
    for action in chunk:
        result = probe.step(action)
        obs = result.observation
        if result.done:
            break
    done = bool(result.done) if result is not None else False
    print(f"step {step}: {N_SAMPLES} muestras (ruido amplio) | {len(chunk)} accion(es) del chunk | done={done}")
    if done:
        print("El entorno termino (done=True); detengo el barrido.")
        break

# 4) Restaurar ruido N(0,1) y CSV por defecto (para no afectar corridas futuras).
model.set_context(noise_scale=[], csv_path="")
print("Listo. CSV ->", BIG_CSV, "| ruido restaurado a N(0,1).")

## 4. Correr los experimentos
Un `LiberoController` por tarea (escenario); `run_experiments` corre los
episodios y graba un video por cada uno. pi0 devuelve un chunk de acciones que
el runner ejecuta una a una.


In [4]:
from core import run_experiments, View
from benchmarks.libero import LiberoController

task_list = LiberoController.tasks(cfg.suite)[:cfg.num_tasks]
benchmarks = [LiberoController(task_id=tid, suite=cfg.suite) for tid, _ in task_list]

results = run_experiments(
    model, benchmarks,
    max_steps=cfg.max_steps,
    episodes_per_task=cfg.episodes_per_task,
    out_dir=cfg.out_dir,
    record_view=View.AGENT,
)
print(f"{len(results)} episodios corridos")


[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /mnt-homes/wapol/asoto/diegoftpxd/miniforge3/envs/openvla/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py (macros.py:55)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[Warning]: datasets path /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation/benchmarks/libero/Libero-10-r/libero/libero/../datasets does not exist!
[Warning]: datasets path /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation/benchmarks/libero/Libero-10-r/libero/libero/../datasets does not exist!


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray._reconstruct])` or the `torch.serialization.safe_globals([numpy.core.multiarray._reconstruct])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch

## 5. Resultados — tabla y tasa de éxito


In [ ]:
import pandas as pd
from core import summarize

df = pd.DataFrame(results)
resumen = summarize(results)
print("Tasa de exito global: {exitos}/{total} = {tasa_exito:.0%}".format(**resumen))
df[["scenario", "instruction", "episode", "success", "steps"]]


In [ ]:
import matplotlib.pyplot as plt

por_escenario = df.groupby("scenario")["success"].mean()
ax = por_escenario.plot(kind="bar", ylim=(0, 1), color="#F58518", rot=0)
ax.set_xlabel("escenario"); ax.set_ylabel("tasa de exito")
ax.set_title("Exito por tarea (pi0)"); plt.tight_layout(); plt.show()


## 6. Resultados — videos


In [ ]:
from IPython.display import Video, display

for r in results:
    if r["video"]:
        print(f"escenario {r['scenario']} | {r['instruction']} | exito={r['success']}")
        display(Video(r["video"], embed=True, width=320))
